In [1]:
import matplotlib
matplotlib.use("Agg")  # headless — must precede any other matplotlib import


# F5 — Backtest de Alerta Temprana

Evalúa si el **sentimiento negativo de las noticias** (clasificado con DeBERTa-v3-base, solo artículos `is_relevant=1`) se **adelanta** a las caídas de precio en los 10 tickers de la cartera.

Se usan únicamente noticias hasta el momento de la señal; los retornos son siempre **futuros** (sin lookahead bias).

In [2]:
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from scipy import stats

from src.backtest.signal  import build_signal
from src.backtest.returns import build_returns
from src.data.store       import DB_PATH

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

FIGURES_DIR = ROOT / "reports" / "figures"
RESULTS_DIR = ROOT / "reports" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["AAPL", "NVDA", "MSFT", "AMZN", "META", "GOOGL", "TSLA", "JPM", "XOM", "PFE"]
print(f"DB: {DB_PATH}")
print(f"Figures -> {FIGURES_DIR}")


DB: C:\Users\rarra\Desktop\TFM\data\processed\radar.db
Figures -> C:\Users\rarra\Desktop\TFM\reports\figures


## 0. Datos: señal de sentimiento y retornos

In [3]:
signal_df  = build_signal(TICKERS)
returns_df = build_returns(TICKERS)

analysis = signal_df.merge(returns_df, on=["ticker", "date"], how="inner")
analysis["date_dt"] = pd.to_datetime(analysis["date"])

print(f"Filas total (ticker × día): {len(analysis):,}")
print(f"Rango fechas: {analysis['date'].min()} → {analysis['date'].max()}")
print(f"Tickers: {sorted(analysis['ticker'].unique())}")
print()
print("Señal raw (pooled, no ponderada):")
print(analysis[["raw", "weighted", "raw_ma3", "raw_ma5"]].describe().round(4))
print()
print("Noticias relevantes por día (cuando n_articles > 0):")
print(analysis[analysis["n_articles"] > 0]["n_articles"].describe().round(1))
print()
# Per-ticker stats
tbl = (
    analysis.groupby("ticker")
    .agg(
        n_days=("date", "count"),
        n_news_days=("n_articles", lambda x: (x > 0).sum()),
        avg_articles=("n_articles", "mean"),
        avg_signal=("raw", "mean"),
    )
    .round(3)
)
print("Por ticker:")
print(tbl)


Filas total (ticker × día): 2,510
Rango fechas: 2025-06-25 → 2026-06-24
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'JPM', 'META', 'MSFT', 'NVDA', 'PFE', 'TSLA', 'XOM']

Señal raw (pooled, no ponderada):
             raw   weighted    raw_ma3    raw_ma5
count  2510.0000  2510.0000  2510.0000  2510.0000
mean      0.1039     0.0997     0.1034     0.1030
std       0.2503     0.2379     0.1869     0.1632
min      -1.0000    -0.9944    -0.6667    -0.6000
25%       0.0000     0.0000     0.0000     0.0000
50%       0.0000     0.0000     0.0000     0.0000
75%       0.1429     0.1462     0.1886     0.1844
max       1.0000     0.9941     0.9333     0.8317

Noticias relevantes por día (cuando n_articles > 0):
count    951.0
mean      12.1
std       12.0
min        1.0
25%        4.0
50%        8.0
75%       16.0
max      113.0
Name: n_articles, dtype: float64

Por ticker:
        n_days  n_news_days  avg_articles  avg_signal
ticker                                               
AAPL       251           62

## A. Lead-lag: ¿La señal adelanta o reacciona al precio?

Cross-correlación entre la señal de sentimiento en el día *t* y el retorno diario en el día *t + k*:

$$\text{corr}(\text{signal}[t],\; \text{daily\_ret}[t+k])$$

- **k > 0** → la señal de hoy predice el retorno de *k* días después (**señal adelanta**).
- **k = 0** → contemporáneo.
- **k < 0** → retornos pasados predicen la señal de hoy (señal reacciona).


In [4]:
LAGS = list(range(-5, 6))

def pooled_leadlag(df, signal_col, ret_col="daily_ret"):
    """Pearson r(signal[t], ret[t+k]) pooled across tickers."""
    results = []
    for k in LAGS:
        sig_vals, ret_vals = [], []
        for ticker, grp in df.groupby("ticker"):
            g = grp.sort_values("date").reset_index(drop=True)
            s = g[signal_col].values
            r = g[ret_col].values
            n = len(s)
            if k >= 0:
                a, b = s[: n - k] if k > 0 else s, r[k:] if k > 0 else r
            else:
                kk = abs(k)
                a, b = s[kk:], r[: n - kk]
            sig_vals.extend(a)
            ret_vals.extend(b)

        sv = np.array(sig_vals, dtype=float)
        rv = np.array(ret_vals, dtype=float)
        mask = np.isfinite(sv) & np.isfinite(rv)
        sv, rv = sv[mask], rv[mask]
        r, p = stats.pearsonr(sv, rv) if len(sv) > 10 else (0.0, 1.0)
        results.append({"lag": k, "corr": r, "pval": p, "n": int(mask.sum())})
    return pd.DataFrame(results)

ll_raw  = pooled_leadlag(analysis, "raw")
ll_ma3  = pooled_leadlag(analysis, "raw_ma3")
ll_ma5  = pooled_leadlag(analysis, "raw_ma5")

print("Lead-lag (señal raw):")
print(ll_raw[["lag", "corr", "pval"]].to_string(index=False))
print()
print("Lead-lag (MA5):")
print(ll_ma5[["lag", "corr", "pval"]].to_string(index=False))


Lead-lag (señal raw):
 lag      corr     pval
  -5  0.025395 0.208916
  -4  0.006711 0.739360
  -3  0.037147 0.064909
  -2  0.042133 0.035896
  -1  0.086783 0.000014
   0  0.072686 0.000275
   1 -0.004555 0.819933
   2 -0.015927 0.426949
   3 -0.021790 0.278044
   4 -0.014622 0.467600
   5  0.018531 0.358247

Lead-lag (MA5):
 lag      corr     pval
  -5  0.062133 0.002092
  -4  0.075603 0.000175
  -3  0.071728 0.000360
  -2  0.054298 0.006838
  -1  0.035347 0.077824
   0  0.003927 0.844423
   1 -0.012939 0.517871
   2 -0.013304 0.506976
   3 -0.011029 0.583021
   4 -0.009461 0.638369
   5 -0.010022 0.619315


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ll, title in [
    (axes[0], ll_raw,  "Señal raw (sin suavizar)"),
    (axes[1], ll_ma5,  "Señal MA5 (5-day moving avg)"),
]:
    colors = ["#d62728" if k > 0 else ("#7f7f7f" if k == 0 else "#aec7e8")
              for k in ll["lag"]]
    ax.bar(ll["lag"], ll["corr"], color=colors, alpha=0.85, edgecolor="white", width=0.7)
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(0.5, color="gray", lw=0.6, ls="--", alpha=0.5)

    for _, row in ll.iterrows():
        if row["pval"] < 0.05:
            y_off = 0.0008 if row["corr"] >= 0 else -0.0025
            ax.text(row["lag"], row["corr"] + y_off, "*", ha="center", va="bottom",
                    fontsize=13, fontweight="bold")

    ax.set_xlabel("Lag k (días)")
    ax.set_ylabel("Correlación de Pearson r")
    ax.set_title(title)
    ax.set_xticks(LAGS)
    ax.set_xticklabels([str(k) for k in LAGS], fontsize=8)

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor="#d62728", alpha=0.85, label="lag>0: señal adelanta al precio"),
    Patch(facecolor="#aec7e8", alpha=0.85, label="lag<0: precio adelanta a la señal"),
    Patch(facecolor="#7f7f7f", alpha=0.85, label="lag=0: contemporáneo"),
]
axes[0].legend(handles=legend_els, fontsize=8)

plt.suptitle("Lead-lag Cross-Correlation — 10 tickers pooled", fontsize=12, y=1.01)
plt.tight_layout()
out_path = FIGURES_DIR / "backtest_leadlag.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")


Guardado: C:\Users\rarra\Desktop\TFM\reports\figures\backtest_leadlag.png


## B. Event Study — Señal Negativa vs. Retornos Futuros

Se define **"día de señal negativa"** como cualquier día en que la señal MA5 cae en el **quintil más bajo** (20% más negativo) de su distribución por ticker.

Se compara la media de los retornos futuros (t+1, t+3, t+5) en días de señal negativa frente al resto de días, y se reporta la diferencia en puntos básicos (1 bp = 0.01%).


In [6]:
results_event = []

for ticker, grp in analysis.groupby("ticker"):
    grp = grp.sort_values("date").copy()
    q20 = grp["raw_ma5"].quantile(0.20)
    grp["neg_signal"] = grp["raw_ma5"] <= q20

    for h in (1, 3, 5):
        ret_col = f"ret_{h}"
        sub = grp.dropna(subset=[ret_col])
        neg = sub[sub["neg_signal"]][ret_col]
        pos = sub[~sub["neg_signal"]][ret_col]
        if len(neg) < 5 or len(pos) < 5:
            continue
        diff = neg.mean() - pos.mean()
        tstat, pval = stats.ttest_ind(neg, pos, equal_var=False)
        results_event.append({
            "ticker": ticker,
            "horizon": h,
            "neg_signal_days": len(neg),
            "other_days": len(pos),
            "ret_neg_bps": round(neg.mean() * 10000, 1),
            "ret_other_bps": round(pos.mean() * 10000, 1),
            "diff_bps": round(diff * 10000, 1),
            "pval": round(pval, 3),
            "significant": pval < 0.10,
        })

event_df = pd.DataFrame(results_event)

# Aggregated (pooled across tickers)
pool_rows = []
for ticker, grp in analysis.groupby("ticker"):
    grp = grp.sort_values("date").copy()
    q20 = grp["raw_ma5"].quantile(0.20)
    grp["neg_signal"] = grp["raw_ma5"] <= q20
    pool_rows.append(grp)

pool = pd.concat(pool_rows)
agg_rows = []
for h in (1, 3, 5):
    ret_col = f"ret_{h}"
    sub = pool.dropna(subset=[ret_col])
    neg = sub[sub["neg_signal"]][ret_col]
    pos = sub[~sub["neg_signal"]][ret_col]
    tstat, pval = stats.ttest_ind(neg, pos, equal_var=False)
    diff = neg.mean() - pos.mean()
    agg_rows.append({
        "ticker": "POOL",
        "horizon": h,
        "neg_signal_days": len(neg),
        "other_days": len(pos),
        "ret_neg_bps": round(neg.mean() * 10000, 1),
        "ret_other_bps": round(pos.mean() * 10000, 1),
        "diff_bps": round(diff * 10000, 1),
        "pval": round(pval, 3),
        "significant": pval < 0.10,
    })

pool_df = pd.DataFrame(agg_rows)

print("=== Event Study — Pool de los 10 tickers ===")
print(pool_df[["horizon", "neg_signal_days", "other_days",
               "ret_neg_bps", "ret_other_bps", "diff_bps", "pval", "significant"]].to_string(index=False))
print()
print("=== Por ticker (horizonte 1d) ===")
h1 = event_df[event_df["horizon"] == 1].sort_values("diff_bps")
print(h1[["ticker", "neg_signal_days", "ret_neg_bps", "ret_other_bps", "diff_bps", "pval", "significant"]].to_string(index=False))


=== Event Study — Pool de los 10 tickers ===
 horizon  neg_signal_days  other_days  ret_neg_bps  ret_other_bps  diff_bps  pval  significant
       1             1321        1179          7.2            5.1       2.1 0.782        False
       3             1317        1163         22.8           14.7       8.1 0.538        False
       5             1304        1156         39.9           26.4      13.5 0.422        False

=== Por ticker (horizonte 1d) ===
ticker  neg_signal_days  ret_neg_bps  ret_other_bps  diff_bps  pval  significant
   JPM               92        -13.2           19.0     -32.2 0.090         True
  MSFT              161        -19.7            3.0     -22.7 0.282        False
  AMZN              159         -2.7           15.6     -18.3 0.507        False
  AAPL              147         12.5           19.0      -6.5 0.725        False
  NVDA              165         10.0           10.6      -0.6 0.985        False
 GOOGL              161         31.4           22.6   

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, h in zip(axes, [1, 3, 5]):
    sub = event_df[event_df["horizon"] == h].sort_values("diff_bps")
    colors = ["#d62728" if v < 0 else "#2ca02c" for v in sub["diff_bps"]]
    ax.barh(sub["ticker"], sub["diff_bps"], color=colors, alpha=0.85, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8)

    # Pool marker
    pool_val = pool_df[pool_df["horizon"] == h]["diff_bps"].values[0]
    ax.axvline(pool_val, color="navy", lw=1.5, ls="--", label=f"Pool avg: {pool_val:.1f} bp")

    for _, row in sub.iterrows():
        if row["significant"]:
            ax.text(row["diff_bps"] + (3 if row["diff_bps"] >= 0 else -3),
                    row["ticker"], "*", ha="center", va="center", fontsize=11, fontweight="bold")

    ax.set_title(f"Retorno diferencial t+{h}d (bp)")
    ax.set_xlabel("Diferencia retorno (bp) [neg_signal vs resto]")
    ax.legend(fontsize=8)

plt.suptitle("Event Study: días de señal más negativa (Q20) vs. resto", fontsize=11, y=1.01)
plt.tight_layout()
out_path = FIGURES_DIR / "backtest_eventstudy.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")


Guardado: C:\Users\rarra\Desktop\TFM\reports\figures\backtest_eventstudy.png


## C. Caso de Estudio Visual: TSLA y NVDA

Se grafica el precio normalizado (base 100 en el inicio) junto con la señal de sentimiento MA5, para ver si el sentimiento giró a negativo **antes** de las caídas de precio.


In [8]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

for ax, ticker in zip(axes, ["TSLA", "NVDA"]):
    sub = analysis[analysis["ticker"] == ticker].sort_values("date").copy()
    # 'close' comes from build_returns — drop rows with missing close
    sub = sub[sub["close"].notna() & (sub["close"] > 0)]
    dates = sub["date_dt"].values

    price_norm = sub["close"].values / sub["close"].values[0] * 100
    sentiment  = sub["raw_ma5"].values

    ax2 = ax.twinx()

    # Shade negative-sentiment regions (background)
    neg_mask = sentiment < 0
    for i in range(len(dates) - 1):
        if neg_mask[i]:
            ax2.axvspan(dates[i], dates[i + 1], alpha=0.06, color="#d62728", zorder=0)

    ax.plot(dates, price_norm, color="#1f77b4", lw=1.8, label="Price (base 100)", zorder=3)
    ax.fill_between(dates, price_norm, alpha=0.08, color="#1f77b4")
    ax.set_ylabel("Normalized price (base 100)", color="#1f77b4")
    ax.tick_params(axis="y", labelcolor="#1f77b4")

    ax2.plot(dates, sentiment, color="#d62728", lw=1.3, alpha=0.85,
             label="Sentiment MA5", zorder=4)
    ax2.axhline(0, color="#d62728", lw=0.6, ls="--", alpha=0.4)
    ax2.set_ylabel("Sentiment signal MA5", color="#d62728")
    ax2.tick_params(axis="y", labelcolor="#d62728")

    # Annotate biggest 1-month drawdown
    closes = sub["close"].values
    best_end = None
    worst_drop = 0.0
    for i in range(len(closes)):
        for j in range(i + 5, min(i + 25, len(closes))):
            drop = (closes[j] - closes[i]) / closes[i]
            if drop < worst_drop:
                worst_drop = drop
                best_end = j
    if best_end is not None:
        ax.annotate(
            f"Drop {worst_drop*100:.1f}%",
            xy=(dates[best_end], price_norm[best_end]),
            xytext=(dates[best_end], price_norm[best_end] * 1.07),
            fontsize=8, color="#d62728",
            arrowprops=dict(arrowstyle="->", color="#d62728", lw=1),
        )

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
    ax.set_title(f"{ticker} - Price vs. Sentiment Signal (MA5)", fontsize=11)

plt.tight_layout()
out_path = FIGURES_DIR / "backtest_casestudy.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


Saved: C:\Users\rarra\Desktop\TFM\reports\figures\backtest_casestudy.png


## Resultados consolidados

In [9]:
results = {
    "description": "F5 Backtest de alerta temprana: sentimiento noticias vs. retornos de precio",
    "dataset": {
        "tickers": TICKERS,
        "n_ticker_days": len(analysis),
        "date_min": analysis["date"].min(),
        "date_max": analysis["date"].max(),
        "n_relevant_articles": int(analysis["n_articles"].sum()),
        "days_with_news": int((analysis["n_articles"] > 0).sum()),
    },
    "lead_lag": {
        "signal": "raw_ma5",
        "description": "corr(signal[t], daily_ret[t+k]) pooled 10 tickers",
        "lags": ll_ma5[["lag", "corr", "pval", "n"]].round(5).to_dict(orient="records"),
        "best_predictive_lag": int(ll_ma5[ll_ma5["lag"] > 0].nlargest(1, "corr")["lag"].values[0]),
        "best_predictive_corr": float(ll_ma5[ll_ma5["lag"] > 0]["corr"].max().round(5)),
        "lag0_corr": float(ll_ma5[ll_ma5["lag"] == 0]["corr"].values[0].round(5)),
    },
    "event_study_pooled": pool_df.to_dict(orient="records"),
    "event_study_by_ticker_h1": h1[["ticker", "neg_signal_days", "ret_neg_bps",
                                    "ret_other_bps", "diff_bps", "pval"]].to_dict(orient="records"),
    "figures": [
        "reports/figures/backtest_leadlag.png",
        "reports/figures/backtest_eventstudy.png",
        "reports/figures/backtest_casestudy.png",
    ],
}

out_path = RESULTS_DIR / "backtest.json"
out_path.write_text(json.dumps(results, indent=2, default=str), encoding="utf-8")
print(f"Guardado: {out_path}")

print()
print("=" * 60)
print("RESUMEN HONESTO DEL BACKTEST")
print("=" * 60)

# Lead-lag assessment
best_fwd_lag = ll_ma5[ll_ma5["lag"] > 0].nlargest(1, "corr").iloc[0]
lag0 = ll_ma5[ll_ma5["lag"] == 0].iloc[0]
print(f"Lead-lag (MA5):")
print(f"  Mejor correlación predictiva (lag>0): lag={int(best_fwd_lag['lag'])}d, r={best_fwd_lag['corr']:.4f} (p={best_fwd_lag['pval']:.3f})")
print(f"  Correlación contemporánea (lag=0):    r={lag0['corr']:.4f} (p={lag0['pval']:.3f})")

# Event study assessment
pool_h1 = pool_df[pool_df["horizon"] == 1].iloc[0]
pool_h5 = pool_df[pool_df["horizon"] == 5].iloc[0]
print(f"Event study (Q20 señal negativa vs. resto):")
print(f"  Diferencial t+1: {pool_h1['diff_bps']:.1f} bp (p={pool_h1['pval']:.3f})")
print(f"  Diferencial t+5: {pool_h5['diff_bps']:.1f} bp (p={pool_h5['pval']:.3f})")

# Interpretation
print()
print("Interpretación:")
if abs(lag0["corr"]) > abs(best_fwd_lag["corr"]):
    print("  La correlacion mas alta es CONTEMPORANEA (lag=0): el sentimiento COINCIDE")
    print("  con el movimiento del precio mas que anticiparlo. Tipico en noticias reactivas.")
else:
    print("  La correlacion mas alta es PREDICTIVA (lag>0): la senal ADELANTA al precio.")
if abs(pool_h1["diff_bps"]) > 5:
    print(f"  El event study muestra una diferencia de {pool_h1['diff_bps']:.1f} bp en t+1,")
    print("  que es economicamente perceptible aunque no siempre estadisticamente robusta.")
else:
    print("  El event study muestra una diferencia pequena (~0): la senal tiene")
    print("  valor cualitativo (detectar noticias negativas) mas que cuantitativo.")


Guardado: C:\Users\rarra\Desktop\TFM\reports\results\backtest.json

RESUMEN HONESTO DEL BACKTEST
Lead-lag (MA5):
  Mejor correlación predictiva (lag>0): lag=4d, r=-0.0095 (p=0.638)
  Correlación contemporánea (lag=0):    r=0.0039 (p=0.844)
Event study (Q20 señal negativa vs. resto):
  Diferencial t+1: 2.1 bp (p=0.782)
  Diferencial t+5: 13.5 bp (p=0.422)

Interpretación:
  La correlacion mas alta es PREDICTIVA (lag>0): la senal ADELANTA al precio.
  El event study muestra una diferencia pequena (~0): la senal tiene
  valor cualitativo (detectar noticias negativas) mas que cuantitativo.
